In [7]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from db_util import Database

db = Database("/Users/chriswarner/TFData.db")

all_results = db.get_all_athlete_results()
relay_results = db.get_all_relay_results()

In [9]:
relay_results

,school_id,event,result,result2,place,athlete_names,school_name,team_name,school_type,meet_type,meet_num,gender,year
0,131,4 x 100 Relay,43.47,43.47,1.0,"Brady King, Kaleb Hayes, Trevion Williamson, L...",Gary West Side,,Public,Sectional,1,Boys,2023
1,149,4 x 100 Relay,43.95,43.95,2.0,"Donte Moore, Travon McCullough, Davion Terry, ...",Hammond Central,,Public,Sectional,1,Boys,2023
2,150,4 x 100 Relay,44.11,44.11,3.0,"Freddy Brown, DaVierre McNair, Montey Hill, Ta...",Hammond Morton,,Public,Sectional,1,Boys,2023
3,244,4 x 100 Relay,44.74,44.74,4.0,"Peyton Mueller, Jordan Jones, Khtab Ishtawi, A...",Munster,,Public,Sectional,1,Boys,2023
4,84,4 x 100 Relay,44.89,44.89,5.0,"Dandre Brown, Juan Price, Derion Morris, Jalma...",East Chicago Central,,Public,Sectional,1,Boys,2023
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5476,327,4 x 800 Relay,9:48.02,588.02,23.0,"Elise Bulaoro, Emily Shepherd, Isabella Soens,...",South Bend Adams,,Public,State,1,Girls,2024
5477,182,4 x 800 Relay,9:48.21,588.21,24.0,"Alexis Byrd, Monica Lorey, Molly Isaacs, Ella ...",Jasper,,Public,State,1,Girls,2024
5478,23,4 x 800 Relay,9:50.66,590.66,25.0,"Hayden McGuire, Katie Kent, Evie Peterson, Ell...",Bloomington North,,Public,State,1,Girls,2024
5479,110,4 x 800 Relay,9:53.45,593.45,26.0,"Emma Campbell, Lindsey Fisher, Lucy Jenks, Gin...",Floyd Central,,Public,State,1,Girls,2024


In [11]:
all_events = ['Discus', 'Pole Vault', 'Shot Put', 'Long Jump', '800 Meters', '4 x 800 Relay', '4 x 400 Relay', '4 x 100 Relay', '300 Hurdles', '110 Hurdles', '100 Hurdles', '3200 Meters', '1600 Meters', '400 Meters', '200 Meters', 'High Jump']


regional_results = all_results[(all_results['meet_type'] == 'Regional')]
#display(regional_results)

regional_relay_results = relay_results[relay_results['meet_type'] == 'Regional']
#regional_relay_results

In [13]:
#EXCLUDES RELAY RESULTS, empty df's for the 3 relays

year = 2023 #FOR A SPECIFIED YEAR; CHANGE THIS VALUE TO CHANGE THE YEAR
field_events = ['Discus', 'Pole Vault', 'Shot Put', 'Long Jump', 'High Jump']
track_events = ['800 Meters', '300 Hurdles', '110 Hurdles', '100 Hurdles', '3200 Meters', '1600 Meters', '400 Meters', '200 Meters']
relay_events = ['4 x 100 Relay', '4 x 400 Relay', '4 x 800 Relay']

fieldNonQualifiers = pd.DataFrame()
trackNonQualifiers = pd.DataFrame()
relayNonQualifiers = pd.DataFrame()

regionalQualifiers = pd.DataFrame()

for meetNumber in range (1, 9): #loops through each region 
    region_results = regional_results[(regional_results['meet_num'] == meetNumber) & (regional_results['year'] == year) & (all_results['result_type'] == 'Final')] #specifies regional meet and the year, gets FINAL results only
    region_relay_results = regional_relay_results[(regional_relay_results['meet_num'] == meetNumber) & (regional_relay_results['year'] == year)]
    
    
    for event in all_events: #loops through each event of the regional meet
        region_event_results = region_results[region_results['event'] == event] #results of specific event in specific regional meet
        male_event_results = region_event_results[region_event_results['gender'] == 'Boys'].sort_values(by=['result2'], ascending=True) #divides by gender
        female_event_results = region_event_results[region_event_results['gender'] == 'Girls'].sort_values(by=['result2'], ascending=True)
        
        region_relay_event_results = region_relay_results[region_relay_results['event'] == event]
        male_relay_event_results = region_relay_event_results[region_relay_event_results['gender'] == 'Boys'].sort_values(by=['result2'], ascending=True)
        female_relay_event_results = region_relay_event_results[region_relay_event_results['gender'] == 'Girls'].sort_values(by=['result2'], ascending=True)
        
        top3Males = pd.DataFrame()
        top3Females = pd.DataFrame()
        
        top3RelayMales = pd.DataFrame()
        top3RelayFemales = pd.DataFrame()
        
        #display(male_event_results)
        #display(female_event_results)
        if (event in field_events and len(male_event_results) != 0): #top individual performers, accounts for ties
            top3Males = male_event_results.tail(3)
                
            i = -4
            while (male_event_results.iloc[-3]['result2'] == male_event_results.iloc[i]['result2']): #accounts for ties for 3rd place
                top3Males = pd.concat([top3Males, male_event_results.iloc[[i]]], ignore_index=True)
                i-=1
                    
            male_event_results = male_event_results.drop(male_event_results.index[-len(top3Males):]) #clears male_event_results of qualifiers
            fieldNonQualifiers = pd.concat([fieldNonQualifiers, male_event_results], ignore_index=True)
            
        if (event in track_events and len(male_event_results) != 0):
            top3Males = male_event_results.head(3)
                
            i = 3
            while (male_event_results.iloc[2]['result2'] == male_event_results.iloc[i]['result2']): #accounts for ties for 3rd place
                top3Males = pd.concat([top3Males, male_event_results.iloc[[i]]], ignore_index=True)
                i+=1
                
            male_event_results = male_event_results.drop(male_event_results.index[:len(top3Males)]) #clears male_event_results of qualifiers
            trackNonQualifiers = pd.concat([trackNonQualifiers, male_event_results], ignore_index=True)
            
        regionalQualifiers = pd.concat([regionalQualifiers, top3Males], ignore_index=True)
            
        if (event in field_events and len(female_event_results) != 0): #top individual performers, accounts for ties
            top3Females = female_event_results.tail(3)
            
            i = -4
            while (female_event_results.iloc[-3]['result2'] == female_event_results.iloc[i]['result2']): #accounts for ties for 3rd place
                top3Females = pd.concat([top3Females, female_event_results.iloc[[i]]], ignore_index=True)
                i-=1
                        
            female_event_results = female_event_results.drop(female_event_results.index[-len(top3Females):]) #clears female_event_results of qualifiers
            fieldNonQualifiers = pd.concat([fieldNonQualifiers, female_event_results], ignore_index=True)
            
        if (event in track_events and len(female_event_results) != 0):
            top3Females = female_event_results.head(3)
                
            i = 3
            while (female_event_results.iloc[2]['result2'] == female_event_results.iloc[i]['result2']): #accounts for ties for 3rd place
                top3Females = pd.concat([top3Females, female_event_results.iloc[[i]]], ignore_index=True)
                i+=1
                
            female_event_results = female_event_results.drop(female_event_results.index[:len(top3Females)]) #clears female_event_results of qualifiers
            trackNonQualifiers = pd.concat([trackNonQualifiers, female_event_results], ignore_index=True)
            
        regionalQualifiers = pd.concat([regionalQualifiers, top3Females], ignore_index=True)
        
        if (event in relay_events and len(male_relay_event_results) != 0):
            top3MaleRelays = male_relay_event_results.head(3)
            
            i = 3
            while (male_relay_event_results.iloc[2]['result2'] == male_relay_event_results.iloc[i]['result2']): #accounts for ties for 3rd place
                top3RelayMales = pd.concat([top3RelayMales, male_relay_event_results.iloc[[i]]], ignore_index=True)
                i+=1
                
            male_relay_event_results = male_relay_event_results.drop(male_relay_event_results.index[:len(top3RelayMales)]) #clears female_event_results of qualifiers
            relayNonQualifiers = pd.concat([relayNonQualifiers, male_relay_event_results], ignore_index=True)
            
        if (event in relay_events and len(female_relay_event_results) != 0):
            top3FemaleRelays = female_relay_event_results.head(3)
            
            i = 3
            while (female_relay_event_results.iloc[2]['result2'] == female_relay_event_results.iloc[i]['result2']): #accounts for ties for 3rd place
                top3RelayFemales = pd.concat([top3RelayFemales, female_relay_event_results.iloc[[i]]], ignore_index=True)
                i+=1
                
            female_relay_event_results = female_relay_event_results.drop(female_relay_event_results.index[:len(top3RelayFemales)]) #clears female_event_results of qualifiers
            relayNonQualifiers = pd.concat([relayNonQualifiers, female_relay_event_results], ignore_index=True)
            
fieldNonQualifiers = fieldNonQualifiers.sort_values(by=['result2'], ascending=True) #sorts nonqualifiers by performance
trackNonQualifiers = trackNonQualifiers.sort_values(by=['result2'], ascending=True)
relayNonQualifiers = relayNonQualifiers.sort_values(by=['result2'], ascending=True)
            
maleFieldNonQualifiers = fieldNonQualifiers[fieldNonQualifiers['gender'] == 'Boys'] #splits nonquals into Boys/Girls
femaleFieldNonQualifiers = fieldNonQualifiers[fieldNonQualifiers['gender'] == 'Girls']

top3MaleFieldNonQual = maleFieldNonQualifiers.tail(3)
top3FemaleFieldNonQual = femaleFieldNonQualifiers.tail(3)

i = -4
while (maleFieldNonQualifiers.iloc[-3]['result2'] == maleFieldNonQualifiers.iloc[i]['result2']): #accounts for ties for 3rd place
    top3MaleFieldNonQual = pd.concat([top3MaleFieldNonQual, maleFieldNonQualifiers.iloc[[i]]], ignore_index=True)
    i-=1

i = -4
while (femaleFieldNonQualifiers.iloc[-3]['result2'] == femaleFieldNonQualifiers.iloc[i]['result2']): #accounts for ties for 3rd place
    top3FemaleFieldNonQual = pd.concat([top3FemaleFieldNonQual, femaleFieldNonQualifiers.iloc[[i]]], ignore_index=True)
    i-=1


maleTrackNonQualifiers = trackNonQualifiers[trackNonQualifiers['gender'] == 'Boys']
femaleTrackNonQualifiers = trackNonQualifiers[trackNonQualifiers['gender'] == 'Girls']

top3MaleTrackNonQual = maleTrackNonQualifiers.head(3)
top3FemaleTrackNonQual = femaleTrackNonQualifiers.head(3)

i = 3
while (maleTrackNonQualifiers.iloc[2]['result2'] == maleTrackNonQualifiers.iloc[i]['result2']): #accounts for ties for 3rd place
    top3MaleTrackNonQual.append(maleTrackNonQualifiers.iloc[i])
    i+=1

i = 3
while (femaleTrackNonQualifiers.iloc[2]['result2'] == femaleTrackNonQualifiers.iloc[i]['result2']): #accounts for ties for 3rd place
    top3FemaleTrackNonQual.append(femaleTrackNonQualifiers.iloc[i])
    top3FemaleTrackNonQual = pd.concat([top3FemaleTrackNonQual, femaleTrackNonQualifiers.iloc[[i]]], ignore_index=True)
    i+=1

maleRelayNonQualifiers = relayNonQualifiers[relayNonQualifiers['gender'] == 'Boys']
femaleRelayNonQualifiers = relayNonQualifiers[relayNonQualifiers['gender'] == 'Girls']

top3MaleRelayNonQual = maleRelayNonQualifiers.head(3)
top3FemaleRelayNonQual = femaleRelayNonQualifiers.head(3)

i = 3
while (maleRelayNonQualifiers.iloc[2]['result2'] == maleRelayNonQualifiers.iloc[i]['result2']): #accounts for ties for 3rd place
    top3MaleRelayNonQual.append(maleRelayNonQualifiers.iloc[i])
    i+=1

i = 3
while (femaleRelayNonQualifiers.iloc[2]['result2'] == femaleRelayNonQualifiers.iloc[i]['result2']): #accounts for ties for 3rd place
    top3FemaleRelayNonQual.append(femaleRelayNonQualifiers.iloc[i])
    top3FemaleRelayNonQual = pd.concat([top3FemaleRelayNonQual, femaleRelayNonQualifiers.iloc[[i]]], ignore_index=True)
    i+=1


regionalQualifiers = pd.concat([regionalQualifiers, top3MaleFieldNonQual, top3FemaleFieldNonQual, top3MaleTrackNonQual, top3FemaleTrackNonQual, top3MaleRelayNonQual, top3FemaleRelayNonQual])

/var/folders/jy/f9gpjn257sb09_jfkrnn8yzr0000gn/T/ipykernel_500/3109825142.py:15: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  region_results = regional_results[(regional_results['meet_num'] == meetNumber) & (regional_results['year'] == year) & (all_results['result_type'] == 'Final')] #specifies regional meet and the year, gets FINAL results only
/var/folders/jy/f9gpjn257sb09_jfkrnn8yzr0000gn/T/ipykernel_500/3109825142.py:15: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  region_results = regional_results[(regional_results['meet_num'] == meetNumber) & (regional_results['year'] == year) & (all_results['result_type'] == 'Final')] #specifies regional meet and the year, gets FINAL results only
/var/folders/jy/f9gpjn257sb09_jfkrnn8yzr0000gn/T/ipykernel_500/3109825142.py:15: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  region_results = regional_results[(regional_results['meet_num'] == meetNumber) 

In [15]:
print(len(regionalQualifiers)) #1242 qualifiers across 2023-2024; 622 for 2023 & 620 for 2024

622


In [17]:
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.width', None)  # Allow automatic width adjustment
pd.set_option('display.max_colwidth', None)

print(len(regionalQualifiers[regionalQualifiers['event']=='High Jump']))
display(regionalQualifiers[regionalQualifiers['event']=='High Jump'])

60


,athlete_id,first,last,gender,event,result_type,grade,result,result2,place,school_name,enrollment,school_type,nickname,host,meet_type,meet_num,year,school_id,athlete_names,team_name
68,44743.0,LUKE,ROBERTSON,Boys,High Jump,Final,JR,"6' 2""",74.00,3.0,Crown Point,2949.0,Public,Bulldogs,Valparaiso HS,Regional,1,2023,72,NaN,NaN
69,48986.0,MIKE,DROHOSKY,Boys,High Jump,Final,SR,"6' 3""",75.00,2.0,Crown Point,2949.0,Public,Bulldogs,Valparaiso HS,Regional,1,2023,72,NaN,NaN
70,48996.0,JAVION,BALLIN,Boys,High Jump,Final,SR,"6' 4""",76.00,1.0,Westville,295.0,Public,Blackhawks,Valparaiso HS,Regional,1,2023,400,NaN,NaN
71,41738.0,JACOB,SANDLIN,Boys,High Jump,Final,SR,"6' 2""",74.00,4.0,Kankakee Valley,1091.0,Public,Kougars,Valparaiso HS,Regional,1,2023,187,NaN,NaN
72,48999.0,SHAUN,FINLEY,Boys,High Jump,Final,SR,"6' 2""",74.00,5.0,Portage,2271.0,Public,Indians,Valparaiso HS,Regional,1,2023,292,NaN,NaN
73,50770.0,MCKENZI,HOLLAMAN,Girls,High Jump,Final,SR,"5' 3""",63.00,3.0,Merrillville,2095.0,Public,Pirates,Portage,Regional,1,2023,226,NaN,NaN
74,50782.0,SAYLOR,GAPINSKI,Girls,High Jump,Final,FR,"5' 4""",64.00,2.0,Valparaiso,2098.0,Public,Vikings,Portage,Regional,1,2023,377,NaN,NaN
75,50764.0,PRECIOUS,KNIGHT,Girls,High Jump,Final,SR,"5' 5""",65.00,1.0,Hammond Bishop Noll,562.0,Private,Warriors,Portage,Regional,1,2023,148,NaN,NaN
76,50775.0,DAEGES,MORGAN,Girls,High Jump,Final,SR,"5' 3""",63.00,3.0,Hobart,1294.0,Public,Brickies,Portage,Regional,1,2023,161,NaN,NaN
77,50783.0,HAILEY,GEISER,Girls,High Jump,Final,SO,"5' 3""",63.00,5.0,Chesterton,2060.0,Public,Trojans,Portage,Regional,1,2023,51,NaN,NaN
